# Topic 4: note2quiz data quality
In Colab, place the project `data` folder in `/content/note2quiz/` before running the notebook.

## Quality gate before checking
| Topic 2 requirement | Data needed | Quality concern | Check | Decision |
|---|---|---|---|---|
| FR1: Read permitted course PDFs | Readable text on selected pages | Completeness and accuracy | Text exists and compare it with the visible PDF | Use or hold |
| FR3 and RAI1: Show support for an answer | File name, PDF page and passage position | Consistency and accuracy | Source hash, page and saved text slice match | Use, repair or hold |

A match to extracted text does not prove that the printed PDF was read correctly.

In [ ]:
%pip -q install pandera pandas matplotlib pypdf langchain-core

In [1]:
from pathlib import Path
import csv, hashlib, json, os, pickle, re
from collections import Counter
import pandas as pd
import pandera.pandas as pa
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from langchain_core.documents import Document
from pypdf import PdfReader

ROOT = Path.cwd()
if not (ROOT / "data" / "prepared_chunks.jsonl").exists():
    ROOT = Path('/content/note2quiz')
DATA, DOCS = ROOT / 'data', ROOT / 'docs'
DOCS.mkdir(parents=True, exist_ok=True)
inspectable_path = DATA / 'prepared_chunks.jsonl'
compat_path = DATA / 'chunks.pkl'
if not inspectable_path.is_file() or not compat_path.is_file():
    raise FileNotFoundError('Topic 3 hand-off is missing: prepared_chunks.jsonl or chunks.pkl')
records = [json.loads(line) for line in inspectable_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print('Topic 3 file:', inspectable_path)
print('Records:', len(records))
assert records, 'Topic 3 hand-off is empty'


Topic 3 file: .\data\prepared_chunks.jsonl
Records: 62


In [2]:
# A1: verify the two Topic 3 hand-off files before checking quality.
with compat_path.open('rb') as handle: loaded_chunks = pickle.load(handle)
assert len(records) == len(loaded_chunks) == 62
assert all(r['text'] == c.page_content and r['record_id'] == c.metadata['record_id'] for r, c in zip(records, loaded_chunks))
print('Hand-off verified:', len(records), 'matching JSONL/PKL records')
print('First:', records[0]['record_id'], records[0]['source'], records[0]['page'])


Hand-off verified: 62 matching JSONL/PKL records
First: S01-p0023-c001 think_python_2e.pdf 23


In [3]:
# A2: quality inventory with project-specific PDF page and checksum checks.
source_files = {name: DATA / name for name in {r['source'] for r in records}}
readers = {name: PdfReader(str(path)) for name, path in source_files.items()}
hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in source_files.items()}
def clean_page(source_id, page_number, raw_text):
    lines = raw_text.replace('\r\n','\n').replace('\r','\n').replace('\f','').split('\n')
    if source_id == 'S01' and lines:
        printed = page_number - 22
        head = lines[0].strip()
        if head == f'{printed} Chapter 1. The way of the program' or re.fullmatch(r'1\.\d+\. .+ ' + str(printed), head):
            lines = lines[1:]
    kept = []
    for line in lines:
        if source_id == 'S02' and re.fullmatch(r'(?:Teamwork in Business\s*\|\s*\d+|\d+\s*\|\s*Teamwork in Business)', line.strip()):
            continue
        kept.append(line.rstrip())
    return '\n'.join(kept).strip('\n')

rows = []
for r in records:
    name, page = r['source'], r['page']
    valid_page = type(page) is int and 1 <= page <= len(readers[name].pages)
    cleaned = clean_page(r['source_id'], page, readers[name].pages[page-1].extract_text() or '') if valid_page else ''
    text = r['text']
    rows.append({
        'record_id': r['record_id'], 'source': name, 'page': page, 'text': text,
        'char_count': len(text.strip()), 'source_ok': source_files[name].is_file(),
        'page_ok': valid_page, 'checksum_ok': r['source_sha256'] == hashes[name],
        'position_ok': valid_page and cleaned[r['start_character']:r['end_character']] == text,
        'bad_char': '\x00' in text or '\ufffd' in text,
        'noisy_whitespace': bool(re.search(r'\n{4,}| {4,}', text)),
        'short_warning': len(text.strip()) < 100,
        'figure_warning': bool(re.search(r'\bFigure\s+\d', text, flags=re.I))
    })
df = pd.DataFrame(rows)
print(df[['source','page','char_count','short_warning','figure_warning']].describe(include='all').to_string())
for field in ['bad_char','noisy_whitespace','short_warning','figure_warning']:
    print(field, int(df[field].sum()))


                     source       page  char_count short_warning figure_warning
count                    62  62.000000   62.000000            62             62
unique                    2        NaN         NaN             2              2
top     think_python_2e.pdf        NaN         NaN         False          False
freq                     39        NaN         NaN            59             60
mean                    NaN  17.806452  415.177419           NaN            NaN
std                     NaN  11.274283  110.639797           NaN            NaN
min                     NaN   2.000000   60.000000           NaN            NaN
25%                     NaN   4.000000  420.250000           NaN            NaN
50%                     NaN  24.000000  450.000000           NaN            NaN
75%                     NaN  27.000000  484.750000           NaN            NaN
max                     NaN  30.000000  499.000000           NaN            NaN
bad_char 0
noisy_whitespace 0
short_warn

In [4]:
# A3: a repeatable 16-record review set: first record on each of 12 pages plus four unusual records.
first = [group.iloc[0]['record_id'] for _, group in df.groupby(['source','page'], sort=True)]
unusual = ['S01-p0023-c002','S01-p0027-c007','S01-p0025-c002','S02-p0005-c004']
review_ids = list(dict.fromkeys(first + unusual))
assert len(review_ids) == 16
for record_id in review_ids:
    r = next(item for item in records if item['record_id'] == record_id)
    print('---', record_id, r['source'], 'PDF page', r['page'], '---')
    print(r['text'])
print('Read these against the PDF pages. Short text is a warning, not automatic deletion.')


--- S02-p0002-c001 teamwork_in_business.pdf PDF page 2 ---
1.  Teamwork in Business
Learning Objectives
• Define a team and describe its key characteristics.
• Explain why organizations use teams and describe different types of teams.
• Explain why teams may be effective or ineffective.
• Identify factors that contribute to team cohesiveness.
• Understand the importance of learning to participate in team-based activities.
• Identify the skills needed by team members and the roles that members of a team might play.
--- S02-p0003-c001 teamwork_in_business.pdf PDF page 3 ---
Figure 1.1: Woman on the phone.
Yet there was a glimmer of hope at Motorola. Despite its recent lapse in
cell phone fashion sense, Motorola still maintained a concept-phone
unit—a group responsible for designing futuristic new product features
such as speech-recognition capability, flexible touchscreens, and touch-
sensitive body covers. In every concept-phone unit, developers engage
in an ongoing struggle to balance 

## Manual inspection notes
| Record | Observation | Decision |
|---|---|---|
| `S01-p0023-c002` | `isproblem` joins words on page 23 | Hold Python passages; extract the text again. |
| `S01-p0025-c001` | `aprint` joins words on page 25 | Same action. |
| `S01-p0026-c001` | `Avalueis` joins words on page 26 | Same action. |
| `S01-p0027-c001` | `strictsyntaxrules` joins words on page 27 | Same action. |
| `S01-p0029-c001` | `statement:An` joins words on page 29 | Same action. |
| `S01-p0030-c001` | `in09` joins words on page 30 | Same action. |
| `S02-p0003-c001` | Figure caption followed by readable text | Keep the text; check questions needing the image. |
| `S02-p0005-c004` | Figure 1.2 is needed for percentage claims | Hold this passage. |

Six of the eight first Python page passages have joined words. The automatic rules did not catch them. The Python PDF text needs to be extracted again before making questions.

In [5]:
# A4: Pandera rules. A matching extracted page does not prove the printed PDF is error-free.
schema = pa.DataFrameSchema({
    'record_id': pa.Column(str, checks=pa.Check(lambda s: s.str.fullmatch(r'S\d{2}-p\d{4}-c\d{3}').fillna(False))),
    'text': pa.Column(str, checks=pa.Check(lambda s: s.str.strip().str.len() > 0)),
    'source': pa.Column(str, checks=pa.Check(lambda s: s.isin(source_files))),
    'page_ok': pa.Column(bool, checks=pa.Check(lambda s: s)),
    'checksum_ok': pa.Column(bool, checks=pa.Check(lambda s: s)),
    'position_ok': pa.Column(bool, checks=pa.Check(lambda s: s)),
    'bad_char': pa.Column(bool, checks=pa.Check(lambda s: ~s)),
    'char_count': pa.Column(int, checks=pa.Check.in_range(1,500))
}, strict=False)
failure_cases = pd.DataFrame()
try:
    schema.validate(df, lazy=True)
    print('Pandera structural rules: 62/62 pass')
except pa.errors.SchemaErrors as exc:
    failure_cases = exc.failure_cases.copy()
    print(failure_cases.to_string(index=False))
assert df['record_id'].is_unique

# Review warnings have a separate decision; a short or figure reference is not an automatic defect.
issues = []
hard_fail_ids = set()
for _, failure in failure_cases.iterrows():
    index = failure.get('index')
    if pd.notna(index) and int(index) in df.index:
        row = df.loc[int(index)]
        hard_fail_ids.add(row.record_id)
        issues.append((row.record_id,'structural failure: '+str(failure.get('column')),row.char_count,'hold and repair before use'))
for _, row in df.iterrows():
    if row.short_warning:
        issues.append((row.record_id,'short passage',row.char_count,'review context before quiz use'))
    if row.figure_warning:
        issues.append((row.record_id,'figure reference',row.char_count,'review whether answer needs unseen figure'))
for record_id, issue, decision in [
    ('S01-p0023-c002','joined words in extracted PDF text','hold for repair against visible PDF'),
    ('S01-p0025-c002','joined words in extracted PDF text','hold for repair against visible PDF'),
    ('S01-p0027-c007','end of definition without subject','hold until combined with preceding text'),
    ('S01-p0030-c002','end of a question without its opening','hold until combined with preceding text'),
    ('S02-p0003-c007','end of a sentence without its opening','hold until combined with preceding text'),
    ('S02-p0003-c001','figure caption followed by readable narrative','retain narrative; review if a question needs the image'),
    ('S02-p0005-c004','figure needed for percentage claims','hold questions needing the figure')
]:
    issues.append((record_id,issue,len(next(r['text'] for r in records if r['record_id']==record_id)),decision))
source_hold_ids = {r['record_id'] for r in records if r['source_id'] == 'S01'}
for r in records:
    if r['record_id'] in source_hold_ids:
        issues.append((r['record_id'],'Python extraction errors found across sampled pages',len(r['text']),'hold until re-extracted and checked'))
failure_path = DOCS / 'week4_validation_failures.csv'
with failure_path.open('w',newline='',encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerow(['record_id','classification','char_count','decision'])
    writer.writerows(issues)
print('Structural failures:',len(failure_cases),'; review rows:',len(issues))
print('Review records:',sorted({item[0] for item in issues}))


Pandera structural rules: 62/62 pass
Structural failures: 0 ; review rows: 51
Review records: ['S01-p0023-c001', 'S01-p0023-c002', 'S01-p0023-c003', 'S01-p0023-c004', 'S01-p0023-c005', 'S01-p0024-c001', 'S01-p0024-c002', 'S01-p0024-c003', 'S01-p0024-c004', 'S01-p0024-c005', 'S01-p0024-c006', 'S01-p0025-c001', 'S01-p0025-c002', 'S01-p0025-c003', 'S01-p0025-c004', 'S01-p0026-c001', 'S01-p0026-c002', 'S01-p0026-c003', 'S01-p0026-c004', 'S01-p0027-c001', 'S01-p0027-c002', 'S01-p0027-c003', 'S01-p0027-c004', 'S01-p0027-c005', 'S01-p0027-c006', 'S01-p0027-c007', 'S01-p0028-c001', 'S01-p0028-c002', 'S01-p0028-c003', 'S01-p0028-c004', 'S01-p0028-c005', 'S01-p0028-c006', 'S01-p0029-c001', 'S01-p0029-c002', 'S01-p0029-c003', 'S01-p0029-c004', 'S01-p0029-c005', 'S01-p0030-c001', 'S01-p0030-c002', 'S02-p0003-c001', 'S02-p0003-c007', 'S02-p0005-c004']


## A5. Review of flagged passages
| Record | Warning and evidence | Decision |
|---|---|---|
| `S01-p0023-c002` | `isproblem` in extracted text; visible PDF has spaces | Hold and extract the text again. |
| `S01-p0027-c007` | Only the end of a definition remains | Hold until joined with preceding text. |
| `S02-p0005-c004` | Percentage claims need Figure 1.2 | Hold until the figure is checked. |
| `S02-p0003-c001` | Caption followed by readable text | Keep, but check questions needing the image. |

These are manual decisions. The Pandera rules passed all 62 records.

In [6]:
# B1: exact duplicate check; preserve every source in the review CSV.
seen, duplicate_rows, unique = {}, [], []
for r in records:
    norm = re.sub(r'\s+', ' ', r['text'].strip()).casefold()
    digest = hashlib.sha256(norm.encode('utf-8')).hexdigest()
    if digest in seen:
        duplicate_rows.append([r['record_id'], seen[digest]['record_id'], r['source'], seen[digest]['source'], digest, 'review before removal'])
    else:
        seen[digest] = r
        unique.append(r)
duplicate_path = DOCS / 'week4_duplicate_review.csv'
with duplicate_path.open('w',newline='',encoding='utf-8') as handle:
    writer=csv.writer(handle)
    writer.writerow(['duplicate_id','first_id','duplicate_source','first_source','normalised_sha256','decision'])
    writer.writerows(duplicate_rows)
print('Exact duplicates:',len(duplicate_rows),'Unique records:',len(unique))


Exact duplicates: 0 Unique records: 62


In [7]:
# B2: coverage by source; counts signal where to investigate, not whether a subject is fairly served.
counts = Counter(r['source'] for r in unique)
provisional_counts = Counter(r['source'] for r in unique if r['record_id'] not in source_hold_ids and r['record_id'] not in {'S02-p0003-c007','S02-p0005-c004'})
fig, ax = plt.subplots(figsize=(7,4))
sources = ['think_python_2e.pdf','teamwork_in_business.pdf']
labels = ['Think Python','Teamwork in Business']
before = [counts[name] for name in sources]
after = [provisional_counts[name] for name in sources]
ax.bar([-0.18,0.82],before,width=0.36,label='Prepared',color='#475569')
ax.bar([0.18,1.18],after,width=0.36,label='After hold',color='#8aa4bd')
ax.set_xticks([0,1],labels)
ax.set_ylabel('Passages')
ax.set_title('note2quiz passages by source')
ax.legend()
for x, y in zip([-0.18,0.82,0.18,1.18],before+after): ax.text(x,y+0.5,str(y),ha='center')
ax.set_ylim(0,46)
fig.tight_layout()
plot_path = DOCS / 'week4_coverage_by_source.png'
fig.savefig(plot_path,dpi=160)
plt.close(fig)
print(counts)
print('After hold:',provisional_counts)
page_counts = Counter((r['source'], r['page']) for r in unique)
for (source, page), count in sorted(page_counts.items()):
    print(source, 'PDF page', page, ':', count, 'passages')
print('Only 12 selected pages; The Foundations of Business is not in this prepared export.')


Counter({'think_python_2e.pdf': 39, 'teamwork_in_business.pdf': 23})
After hold: Counter({'teamwork_in_business.pdf': 21})
teamwork_in_business.pdf PDF page 2 : 5 passages
teamwork_in_business.pdf PDF page 3 : 7 passages
teamwork_in_business.pdf PDF page 4 : 7 passages
teamwork_in_business.pdf PDF page 5 : 4 passages
think_python_2e.pdf PDF page 23 : 5 passages
think_python_2e.pdf PDF page 24 : 6 passages
think_python_2e.pdf PDF page 25 : 4 passages
think_python_2e.pdf PDF page 26 : 4 passages
think_python_2e.pdf PDF page 27 : 7 passages
think_python_2e.pdf PDF page 28 : 6 passages
think_python_2e.pdf PDF page 29 : 5 passages
think_python_2e.pdf PDF page 30 : 2 passages
Only 12 selected pages; The Foundations of Business is not in this prepared export.


## B2. Coverage by study area
The area labels below come from the book titles. They are a rough guide, not topic labels for each passage.

| Area | Prepared | After hold | Gap |
|---|---:|---:|---|
| Computing | 39 | 0 | Python text needs to be extracted again. |
| Teamwork | 23 | 21 | Only selected pages were checked. |
| Wider business | 0 | 0 | The third PDF was not prepared in Topic 3. |

The lower number of teamwork passages alone does not show unfairness. The missing computing passages after the hold are a real gap for computing questions.

## B3. How data problems could reach a question
| Risk | Source or preparation | Prepared data | Later search and question | Action and test |
|---|---|---|---|---|
| Old Python example | 2015 book includes Python 2 | Version label can be separated from an example | A question may treat old syntax as current | Check course version and generated questions. |
| Missing topics | Only selected pages from two books | Some topics have no passage | Search may choose weak evidence; a question may lack support | Add the third PDF and review questions by topic. |
| Joined words and figures | PDF extraction and text splitting | Words join or a figure is missing | Bad text may appear in an answer | Re-extract and compare with visible pages. |

These are risks to test later, not results about how the finished app treats students.

## B4. Privacy, rights and updates
| Check | Finding | Action |
|---|---|---|
| Source and version | Recorded in Week 3 inventory and manifest | Check new or changed PDFs. |
| Rights | Book licences known; figure exceptions need review | Check figure credits before reuse. |
| Personal data | No accounts, marks or private notes in this sample | Check future uploaded notes. |
| Consent | Not needed for this public book sample | Get permission for private notes. |
| Updates | Course version not confirmed | Confirm it and replace old exports. |

In [8]:
# C2: hold Python passages until re-extraction, plus two teamwork passages.
# Source data stays intact; hold decisions remain in the report.
held = {record_id: 'Python extraction needs repair' for record_id in source_hold_ids}
held.update({
    'S02-p0003-c007': 'sentence fragment',
    'S02-p0005-c004': 'missing figure context'
})
accepted = [r for r in unique if r['record_id'] not in held and r['record_id'] not in hard_fail_ids]
clean_jsonl = DATA / 'prepared_chunks_clean.jsonl'
with clean_jsonl.open('w',encoding='utf-8') as handle:
    for r in accepted: handle.write(json.dumps(r,ensure_ascii=False)+'\n')
clean_chunks = [Document(page_content=r['text'],metadata={k:v for k,v in r.items() if k!='text'}) for r in accepted]
clean_pkl = DATA / 'chunks_clean.pkl'
with clean_pkl.open('wb') as handle: pickle.dump(clean_chunks,handle)
with clean_pkl.open('rb') as handle: reloaded = pickle.load(handle)
reloaded_json = [json.loads(line) for line in clean_jsonl.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(reloaded) == len(reloaded_json) == len(accepted)
assert all(c.page_content == r['text'] and c.metadata['record_id']==r['record_id'] for c,r in zip(reloaded,reloaded_json))
print('Source records:',len(records),'Held:',len(held),'Filtered set:',len(accepted))
print('Held IDs:',held)
print('Clean JSONL/PKL counts and contents match')


Source records: 62 Held: 41 Filtered set: 21
Held IDs: {'S01-p0029-c001': 'Python extraction needs repair', 'S01-p0027-c004': 'Python extraction needs repair', 'S01-p0027-c001': 'Python extraction needs repair', 'S01-p0026-c003': 'Python extraction needs repair', 'S01-p0023-c004': 'Python extraction needs repair', 'S01-p0027-c003': 'Python extraction needs repair', 'S01-p0026-c002': 'Python extraction needs repair', 'S01-p0027-c005': 'Python extraction needs repair', 'S01-p0030-c002': 'Python extraction needs repair', 'S01-p0028-c005': 'Python extraction needs repair', 'S01-p0024-c001': 'Python extraction needs repair', 'S01-p0023-c005': 'Python extraction needs repair', 'S01-p0024-c005': 'Python extraction needs repair', 'S01-p0025-c002': 'Python extraction needs repair', 'S01-p0029-c003': 'Python extraction needs repair', 'S01-p0024-c003': 'Python extraction needs repair', 'S01-p0023-c003': 'Python extraction needs repair', 'S01-p0023-c001': 'Python extraction needs repair', 'S01-p00

# Summary
- 62 input passages: 39 Python and 23 teamwork. All passed the automatic rules; no exact duplicates were found. Three are shorter than 100 characters and two mention a figure.
- Joined words were found on six of eight sampled Python pages. All 39 Python passages are held until the PDF text is extracted again and checked. Two teamwork passages are also held.
- The 21 remaining teamwork passages are saved for checking only. They have not all been compared with the visible pages.
- Decision: HOLD for making questions. See `docs/week4_data_quality_report.md`.